# GLO-NCA: Robust 3D Segmentation with Built-in Quality Control
### John Kalkhof, Anirban Mukhopadhyay
__https://arxiv.org/pdf/2309.02954.pdf__



***

## __The Backbone Model__
<div>
<img src="src/images/model_M3DNCA.png" width="600"/>
</div>

## _1. Imports_

In [ ]:
import torch
from src.datasets.Nii_Gz_Dataset_3D import Dataset_NiiGz_3D_BraTS
from src.models.Model_BasicNCA3D import BasicNCA3D
from src.losses.LossFunctions import DiceCELoss
from src.utils.Experiment import Experiment
from src.agents.Agent_GLO_NCA import Agent_GLO_NCA

## _2. Configure experiment_
- __AutoReload__
    - If an experiment already exists in _model\_path_ the config will __always__ be overwritten with the existing one
    - Additionally if the model has been saved previously, this state will be reloaded
- Download _prostate_ data from 'http://medicaldecathlon.com/' and adapt 'img_path' and 'label_path'
- If you want to change the number of downscaling levels you need to:
    - Set __train_model__ to n_level-1. 
    - Define __inference_steps__ for each level [x, y, z]
    - Define __input_size__ for each level [[], [], ...]

In [ ]:
# Global Context-Aware GLO-NCA on multi-modal BraTS.
# Point BOTH paths to the BraTS dataset root (one sub-folder per patient).
config = [{
    'img_path':   r"PATH_TO_BRATS_ROOT",
    'label_path': r"PATH_TO_BRATS_ROOT",
    'model_path': r"runs/GLO_NCA_BraTS",
    'device': "cuda:0",
    'unlock_CPU': True,
    # Optimizer
    'optimizer': 'adamw',
    'lr': 16e-4,
    'lr_gamma': 0.9999,
    'betas': (0.9, 0.99),
    'weight_decay': 1e-4,
    # Training
    'save_interval': 10,
    'evaluate_interval': 10,
    'n_epoch': 1000,
    'batch_size': 1,
    'batch_duplication': 1,
    # Model
    'channel_n': 16,
    'inference_steps': [10, 10],
    'cell_fire_rate': 0.5,
    'input_channels': 4,       # T1, T1ce, T2, FLAIR
    'output_channels': 3,      # WT, TC, ET
    'hidden_size': 64,
    'train_model': 1,
    'use_attention': True,     # SE global-context block (thesis novelty)
    # Data
    'input_size': [(28, 28, 20), (56, 56, 40)],
    'scale_factor': 2,
    'data_split': [0.7, 0.0, 0.3],
    'keep_original_scale': True,
    'rescale': True,
    'patchify': True,
    'priotize_masks': 0.5,
}]

## _3. Choose architecture, dataset and training agent_

- _Dataset\_Nii\_Gz\_3D_ loads 3D files. If you pass a _slice_ it will be split along the according axis.

In [ ]:
dataset = Dataset_NiiGz_3D_BraTS()
device = torch.device(config[0]['device'])
ca1 = BasicNCA3D(config[0]['channel_n'], config[0]['cell_fire_rate'], device,
                 hidden_size=config[0]['hidden_size'], kernel_size=7,
                 input_channels=config[0]['input_channels'],
                 use_attention=config[0]['use_attention']).to(device)
ca2 = BasicNCA3D(config[0]['channel_n'], config[0]['cell_fire_rate'], device,
                 hidden_size=config[0]['hidden_size'], kernel_size=3,
                 input_channels=config[0]['input_channels'],
                 use_attention=config[0]['use_attention']).to(device)
ca = [ca1, ca2]
agent = Agent_GLO_NCA(ca)
exp = Experiment(config, dataset, ca, agent)
dataset.set_experiment(exp)
exp.set_model_state('train')
data_loader = torch.utils.data.DataLoader(dataset, shuffle=True, batch_size=exp.get_from_config('batch_size'))
loss_function = DiceCELoss()
print("Trainable parameters:", sum(p.numel() for m in ca for p in m.parameters() if p.requires_grad))

## _4. Run training_

In [ ]:
agent.train(data_loader, loss_function)

## _5. Evaluate test data_

In [ ]:
agent.getAverageDiceScore(pseudo_ensemble=True, showResults=True)